# Beach robot Double DQN curriculum

This notebook runs the shared three-stage curriculum. Training, evaluation, checkpointing, and resume behavior live in `robots_Q_learning.py`.

In [ ]:
from pathlib import Path
import json
import sys
import matplotlib.pyplot as plt
import torch

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'DeepQBeachRobots').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'DeepQBeachRobots').is_dir():
    raise RuntimeError(f'Cannot locate project root from {Path.cwd()}')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from DeepQBeachRobots.robots_Q_learning import DEFAULT_CURRICULUM, DqnConfig, train

SEED = 7
CONFIG = DqnConfig()
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Config:', CONFIG)
[(stage.name, stage.training_environment) for stage in DEFAULT_CURRICULUM]

In [ ]:
OUTPUT_DIRECTORY = Path('training_runs') / 'curriculum'
agent, result = train(
    total_environment_steps=200_000,
    output_directory=OUTPUT_DIRECTORY,
    seed=SEED,
    curriculum=DEFAULT_CURRICULUM,
    config=CONFIG,
    require_gpu=True,
)
result['best_checkpoint_path']

In [ ]:
metrics = json.loads((OUTPUT_DIRECTORY / 'training_metrics.json').read_text())
print('Final test:', metrics['final_test'])
print('Baselines:', metrics['baselines'])
print('Throughput:', metrics['environment_steps_per_second'], 'environment steps/s')
metrics['stage_history']

In [ ]:
evaluations = metrics['evaluations']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for stage in sorted({entry['stage'] for entry in evaluations}):
    entries = [entry for entry in evaluations if entry['stage'] == stage]
    steps = [entry['environment_steps'] for entry in entries]
    axes[0].plot(steps, [entry['average_picked'] for entry in entries], marker='o', label=stage)
    axes[1].plot(steps, [entry['completion_rate'] for entry in entries], marker='o', label=stage)
    axes[2].plot(steps, [entry['average_failed_moves'] for entry in entries], marker='o', label=stage)
axes[0].set(title='Rubbish picked', xlabel='Environment step', ylabel='Average')
axes[1].set(title='Completion rate', xlabel='Environment step', ylabel='Rate')
axes[2].set(title='Failed moves', xlabel='Environment step', ylabel='Average')
for axis in axes:
    axis.legend()
plt.tight_layout()